In [1]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["harrison worked at kensho"],
    embedding=OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

# The prompt expects input with keys for "context" and "question"
prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI()

retrieval_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
    }
    | prompt
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("where did harrison work?")

'Harrison worked at Kensho.'

In [2]:
from operator import itemgetter

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["harrison worked at kensho"],
    embedding=OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer in the following language: {language}
"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | model
    | StrOutputParser()
)

chain.invoke(
    {
        "question": "where did harrison work",
        "language": "italian",
    }
)

'harrison ha lavorato a kensho'

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI

model = ChatOpenAI()
joke_chain = (
    ChatPromptTemplate.from_template(
        "tell me a joke about {topic}",
    )
    | model
)
poem_chain = (
    ChatPromptTemplate.from_template(
        "write a 2-line poem about {topic}",
    )
    | model
)

map_chain = RunnableParallel(
    joke=joke_chain,
    poem=poem_chain,
)

map_chain.invoke(
    {
        "topic": "bear",
    }
)

{'joke': AIMessage(content='Why did the bear bring a flashlight to the party? \n\nBecause he heard it was going to be a "beary" good time!', response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 13, 'total_tokens': 41}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-ed32f11b-c3bc-4b4e-940f-995ce408767e-0', usage_metadata={'input_tokens': 13, 'output_tokens': 28, 'total_tokens': 41}),
 'poem': AIMessage(content='In the forest deep, a bear roams free,\nStrength and grace, a sight to see.', response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 15, 'total_tokens': 35}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-94a7b260-5f14-4cf7-831b-8e2da26de22d-0', usage_metadata={'input_tokens': 15, 'output_tokens': 20, 'total_tokens': 35})}

In [4]:
%%timeit

joke_chain.invoke(
    {
        "topic": "bear",
    }
)

1.07 s ± 74.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
%%timeit

poem_chain.invoke(
    {
        "topic": "bear",
    }
)

1.03 s ± 169 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit

map_chain.invoke(
    {
        "topic": "bear",
    }
)

1 s ± 155 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
